In [ ]:
import pandas as pd
import boto3

In [ ]:
# Tratamento local do csv bruto

df = pd.read_csv("Vitimas_DadosAbertos_20250112.csv", sep=";", usecols=[2, 3, 7, 8, 16])

# Tratamento data
df["data_acidente"] = pd.to_datetime(df["data_acidente"])
feriados = {(1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (12, 25)}
df = df[df["data_acidente"].dt.month.combine(df["data_acidente"].dt.day, lambda m, d: (m, d)).isin(feriados)]

# Tratamento gênero
df["genero"] = df["genero"].replace({
    "MASCULINO": "M",
    "FEMININO": "F",
    r"NAO INFORMADO|DESCONHECIDO": "NA"
}, regex=True)

# Tratamento idade
df["faixa_idade"] = df["faixa_idade"].replace({
    "NAO INFORMADO": "NA",
    r"MAIOR QUE 80": "80+",
    r"ENTRE\s|\sANOS": "",
    r"\sE\s": "-"
}, regex=True)


df.to_csv("Vitimas_tratado.csv", index=False)

In [ ]:
# Upload do csv tratado para um bucket da AWS

AWS_PROFILE = "localdesafio4"
BUCKET_NAME = "dados-renavam"
CSV_FILE = "Sprint4\Desafio\Vitimas_tratado.csv"
OBJETO_S3 = "dados_acidentes.csv"


session = boto3.Session(profile_name=AWS_PROFILE)
s3_client = session.client("s3")

try:
    s3_client.create_bucket(
        Bucket=BUCKET_NAME,
    )
    print(f"Bucket '{BUCKET_NAME}' criado com sucesso!")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{BUCKET_NAME}' já existe.")

try:
    s3_client.upload_file(CSV_FILE, BUCKET_NAME, OBJETO_S3)
    print(f"Arquivo '{CSV_FILE}' enviado para '{BUCKET_NAME}/{OBJETO_S3}' com sucesso!")
except Exception as e:
    print("Erro ao enviar arquivo:", e)